In [37]:
# Import necessary libraries
import pyarrow.feather as feather
import pandas as pd
import talib
import numpy as np
from pathlib import Path

# Define the Block class
class Block:
    def __init__(self, start_date, end_date, start_price, end_price, duration, data_segment):
        self.start_date = start_date
        self.end_date = end_date
        self.start_price = start_price
        self.end_price = end_price
        self.duration = duration
        self.data_segment = data_segment  # Store all the data points in this block
        self.direction = 'UP' if start_price < end_price else 'DOWN'
        self.features = {}

# Function to load cryptocurrency data
def load_crypto_data(data_file_path, start_date, end_date):
    """
    Load cryptocurrency data, filter by date range, and calculate TEMA and trend information.
    
    Parameters:
    data_file_path (str): Path to the Feather file containing the dataset.
    start_date (str): The start date for filtering the data in 'YYYY-MM-DD' format.
    end_date (str): The end date for filtering the data in 'YYYY-MM-DD' format.
    
    Returns:
    pd.DataFrame: DataFrame containing the filtered data with TEMA, trend, and trend change information.
    """
    # Load data from Feather file
    crypto_df = feather.read_feather(data_file_path)

    # Convert input dates to datetime format
    start_date = pd.to_datetime(start_date).tz_localize('UTC')
    end_date = pd.to_datetime(end_date).tz_localize('UTC')
    
    # Filter data between the start and end dates
    crypto_df = crypto_df[(crypto_df['date'] >= start_date) & (crypto_df['date'] <= end_date)]

    # Calculate the Triple Exponential Moving Average (TEMA) with a default period of 50
    tema_period = 50
    crypto_df['tema'] = talib.TEMA(crypto_df['close'], timeperiod=tema_period)

    # Determine the trend direction (UP, DOWN, STABLE)
    crypto_df['trend'] = np.where(crypto_df['tema'] > crypto_df['tema'].shift(1), 'UP',
                                  np.where(crypto_df['tema'] < crypto_df['tema'].shift(1), 'DOWN', 'STABLE'))

    # Identify significant trend changes (ignoring 'STABLE' transitions)
    crypto_df['is_trend_change'] = crypto_df['trend'] != crypto_df['trend'].shift(1)
    crypto_df['is_significant_trend_change'] = crypto_df['is_trend_change'] & (crypto_df['trend'] != 'STABLE')

    # Assign a unique group ID to each continuous trend segment
    crypto_df['group_id'] = crypto_df['is_significant_trend_change'].cumsum()

    return crypto_df

# Function to create blocks from the data
def create_blocks(crypto_df):
    """
    Create a list of Block objects from the DataFrame.
    """
    blocks = []

    # Group the data by 'group_id'
    grouped = crypto_df.groupby('group_id')

    # Iterate over each group and create a block
    for group_id, group_data in grouped:
        start_date = group_data['date'].iloc[0]
        end_date = group_data['date'].iloc[-1]
        start_price = group_data['close'].iloc[0]
        end_price = group_data['close'].iloc[-1]
        duration = len(group_data)

        # Create a block object that includes all the data points
        block = Block(
            start_date=start_date,
            end_date=end_date,
            start_price=start_price,
            end_price=end_price,
            duration=duration,
            data_segment=group_data  # Store the entire segment of data
        )

        # Add the block to the list
        blocks.append(block)

    return blocks

# Example usage (for debugging purposes):
data_file_path = '/allah/freqtrade/user_data/data/binance/futures/ETH_USDT_USDT-3m-futures.feather'
start_date = '2024-01-01'
end_date = '2024-10-22'

# Load the data
crypto_df = load_crypto_data(data_file_path, start_date, end_date)

# Create blocks
blocks = create_blocks(crypto_df)


In [ ]:
len(blocks)

In [ ]:
blocks[-1].data_segment

In [ ]:
import pandas as pd
import numpy as np
import talib  # For RSI and MACD
from tqdm import tqdm  # Import the progress bar

def compute_rsi(close_prices, period=14):
    """Compute RSI for a given period."""
    return talib.RSI(close_prices, timeperiod=period)

def compute_macd(close_prices):
    """Compute MACD and signal line."""
    macd, macd_signal, macd_hist = talib.MACD(close_prices, fastperiod=12, slowperiod=26, signalperiod=9)
    return macd, macd_signal, macd_hist

def calculate_statistical_features(series):
    """Compute statistical features like mean, std, variance, skewness, etc."""
    return {
        'mean': np.mean(series),
        'median': np.median(series),
        'std': np.std(series),
        'variance': np.var(series),
        'skewness': series.skew(),
        'kurtosis': series.kurtosis(),
        'min': np.min(series),
        'max': np.max(series)
    }

def extract_ts_features(block):
    # Initialize feature dictionary
    features = {}

    # Add manually calculated feature: the length of the block (duration)
    features['length'] = block.duration

    # Get OHLCV data
    close_prices = block.data_segment['close'].values
    volume = block.data_segment['volume'].values

    # Proceed only if the data is non-empty
    if len(close_prices) > 0:
        # Calculate RSI over the period
        rsi_values = compute_rsi(close_prices)
        rsi_stat_features = calculate_statistical_features(pd.Series(rsi_values))

        # Calculate MACD and signal
        macd, macd_signal, macd_hist = compute_macd(close_prices)
        macd_stat_features = calculate_statistical_features(pd.Series(macd))

        # Statistical features for close prices and volume
        close_stat_features = calculate_statistical_features(block.data_segment['close'])
        volume_stat_features = calculate_statistical_features(block.data_segment['volume'])

        # Merge all statistical features into the feature dictionary
        features.update({f'close_{k}': v for k, v in close_stat_features.items()})
        features.update({f'volume_{k}': v for k, v in volume_stat_features.items()})
        features.update({f'rsi_{k}': v for k, v in rsi_stat_features.items()})
        features.update({f'macd_{k}': v for k, v in macd_stat_features.items()})

        # Adding other features like mean, min, max for MACD histogram and signal
        features['macd_hist_mean'] = np.mean(macd_hist)
        features['macd_hist_std'] = np.std(macd_hist)
        features['macd_signal_mean'] = np.mean(macd_signal)
        features['macd_signal_std'] = np.std(macd_signal)
        
        # Example: Count RSI above certain thresholds (e.g., overbought or oversold)
        features['rsi_above_70'] = np.sum(rsi_values > 70)
        features['rsi_below_30'] = np.sum(rsi_values < 30)

    return features

# Apply the feature extraction to each block with a progress bar
for block in tqdm(blocks, desc="Extracting Features"):
    block.features = extract_ts_features(block)

# Output the features of the last block for inspection
print(blocks[-1].features)


In [ ]:
blocks[-1].features

In [ ]:
blocks[-22].data_segment

In [ ]:
import pandas as pd
import mplfinance as mpf

# Prepare the data in the correct format
data_segment = blocks[-1].data_segment  # Assume this is the OHLCV data

data_segment['date'] = pd.to_datetime(data_segment['date'])

# Set the 'date' as the index
data_segment.set_index('date', inplace=True)

# Rename the columns to match mplfinance requirements
data_segment.rename(columns={
    'open': 'Open', 
    'high': 'High', 
    'low': 'Low', 
    'close': 'Close', 
    'volume': 'Volume'
}, inplace=True)

# Plot the OHLCV data as candlesticks
mpf.plot(
    data_segment, 
    type='candle', 
    volume=True,  # Include volume subplot
    style='charles',  # Choose a style (optional)
    title=f'Candlestick Chart from {data_segment.index.min().date()} to {data_segment.index.max().date()}',
    ylabel='Price (USD)',
    ylabel_lower='Volume',
    figsize=(14, 8)
)

# plot last 5 block 

# for i in range(5):
#     data_segment = blocks[-i].data_segment  # Assume this is the OHLCV data
#     data_segment['date'] = pd.to_datetime(data_segment['date'])

#     # Set the 'date' as the index
#     data_segment.set_index('date', inplace=True)

#     # Rename the columns to match mplfinance requirements
#     data_segment.rename(columns={
#         'open': 'Open', 
#         'high': 'High', 
#         'low': 'Low', 
#         'close': 'Close', 
#         'volume': 'Volume'
#     }, inplace=True)

#     # Plot the OHLCV data as candlesticks
#     mpf.plot(
#         data_segment, 
#         type='candle', 
#         volume=True,  # Include volume subplot
#         style='charles',  # Choose a style (optional)
#         title=f'Candlestick Chart from {data_segment.index.min().date()} to {data_segment.index.max().date()}',
#         ylabel='Price (USD)',
#         ylabel_lower='Volume',
#         figsize=(14, 8)
    # )


In [5]:
import numpy as np

# List to store blocks with NaN values and their information
blocks_with_nan = []

# Check for NaN values in the filtered features and replace them with 0
for block in blocks:
    nan_features = {}
    
    # Loop through each feature in the block
    for key, value in block.features.items():
        if isinstance(value, list):  # If the value is a list (from tsfresh), check inside the list
            value = [0 if np.isnan(v) else v for v in value]  # Replace NaN values with 0 in the list
            block.features[key] = value  # Update the feature with NaN replaced
            if any(v == 0 for v in value):  # If any value was replaced, log it
                nan_features[key] = value
        else:
            if np.isnan(value):  # If it's a single value and NaN, replace with 0
                block.features[key] = 0
                nan_features[key] = 0
    
    # If NaN features were found and replaced, save the block info (start_date, duration, and the features containing NaN)
    if nan_features:
        block_info = {
            'start_date': block.start_date,
            'duration': block.duration,
            'nan_features': nan_features
        }
        blocks_with_nan.append(block_info)

# Now `blocks_with_nan` contains all the blocks where NaN values were found and replaced with 0


In [ ]:
blocks_with_nan

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
Y = [blocks[i].duration for i in range(len(blocks) - 1)]
k = 50

X = [[blocks[i - k + j].features for j in range(k)] for i in range(k, len(blocks) - 1)]

print(f'Length of X: {len(X)}, Length of Y: {len(Y)}')

# Initialize a dictionary to store the counts for each threshold l
counts = {}

# Iterate over l from 1 to 20
for l in range(1, 21):
    # Generate the binary Y values based on the current l
    binary_Y = [1 if y > l else 0 for y in Y]
    
    # Count the number of 1s and 0s
    unique, unique_counts = np.unique(binary_Y, return_counts=True)
    
    # Store the counts in the dictionary
    counts[l] = dict(zip(unique, unique_counts))
    
    # Print the count for each unique value in binary_Y
    print(f'For l = {l}: {counts[l]}')

# Prepare data for plotting
l_values = list(counts.keys())
count_0 = [counts[l].get(0, 0) for l in l_values]  # Counts for Y = 0
count_1 = [counts[l].get(1, 0) for l in l_values]  # Counts for Y = 1

# Plot the counts for visualization
plt.figure(figsize=(10, 5))
plt.plot(l_values, count_0, label='Count of Y = 0', marker='o', color='blue')
plt.plot(l_values, count_1, label='Count of Y = 1', marker='o', color='green')

# Add labels and title
plt.xlabel('Threshold l')
plt.ylabel('Count')
plt.title('Distribution of Binary Y Values Based on Threshold l')

# Add legend and grid
plt.legend()
plt.grid(True)

# Show the plot
plt.show()

# Display the distribution of unique Y values in the original data
unique, counts = np.unique(Y, return_counts=True)
print(dict(zip(unique, counts)))

In [ ]:
# This cell is for Y generation. Y is the length of the next block
Y = [blocks[i].duration for i in range(len(blocks) - 1)]

# Generate X: sequences of features from the previous k blocks in correct order
k = 50
X = [[blocks[i - k + j].features for j in range(k)] for i in range(k, len(blocks) - 1)]

# Align Y with X
Y = Y[k:]

Y = [1 if y > 3 else 0 for y in Y]

# Y = 1 if the length of the next block is greater than l
# Y = [1 if y > l else 0 for y in Y]
# Check lengths
len(X), len(Y)


In [ ]:
import numpy as np

# Define the set of all possible features
all_features = [
    'length',
    'close__mean_change', 'close__variance', 'close__standard_deviation', 
    'close__skewness', 'close__kurtosis',
    'volume__mean_change', 'volume__variance', 'volume__skewness',
    'volume__kurtosis', 'volume__mean', 'volume__abs_energy', 'volume__median'
]

# Function to convert a block to a flat list of feature values
def block_to_feature_vector(block):
    feature_vector = []
    for feature in all_features:
        # Extract the value for each feature, or use 0 if the feature is missing
        value = block.get(feature, 0)
        
        # If the value is a list (e.g., [value]), extract the first item from the list
        if isinstance(value, list):
            value = value[0]
        
        feature_vector.append(value)
    
    return feature_vector

# Now convert X (list of sequences) into X_numeric (list of lists of feature vectors)
X_numeric = []

for sequence in X:  # Each sequence is a list of blocks
    sequence_numeric = []
    
    for block in sequence:  # Each block is a dictionary of features
        # Convert block dictionary to a feature vector
        feature_vector = block_to_feature_vector(block)
        sequence_numeric.append(feature_vector)
    
    # Append the numeric sequence to X_numeric
    X_numeric.append(sequence_numeric)

# Convert X_numeric to a numpy array for LSTM input
X_numeric = np.array(X_numeric)

# Reshape X to match the LSTM input shape: (samples, time steps, features)
# X_numeric has the shape (number of samples, time steps, number of features)
X_lstm = X_numeric.reshape((X_numeric.shape[0], X_numeric.shape[1], X_numeric.shape[2]))

# Check the shape of the prepared data
print("Shape of X_lstm:", X_lstm.shape)  # Expected: (number of samples, sequence length, number of features)


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import TensorBoard
import datetime
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# Convert the processed X_lstm to numpy arrays (if not already done)
X = X_lstm

# Reshape X to 2D for scaling
X_reshaped = X.reshape(-1, X.shape[2])

# Standardize the data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_reshaped)

# Reshape back to original shape for LSTM input
X_scaled = X_scaled.reshape(X.shape[0], X.shape[1], X.shape[2])

# Convert Y to numpy array
Y = np.array(Y)

print("Shape of X:", X_scaled.shape)  # Shape of the input data (samples, time steps, features)
print("Shape of Y:", Y.shape)  # Shape of the output labels

# Build the LSTM model
model = models.Sequential()

# First LSTM layer with 50 units and return sequences for the next LSTM layer
model.add(layers.LSTM(units=50, input_shape=(X_scaled.shape[1], X_scaled.shape[2]), return_sequences=True))

# Second LSTM layer with 50 units and return sequences for the next LSTM layer
model.add(layers.LSTM(units=10, return_sequences=False))

# Dropout layer to prevent overfitting (drop 20% of neurons)
model.add(layers.Dropout(0.5))

# Output layer with 1 unit for binary classification (sigmoid activation for binary output)
model.add(layers.Dense(1, activation='sigmoid'))

# Compile the model with Adam optimizer and binary cross-entropy loss for binary classification
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Print the model summary to check the architecture
model.summary()

# TensorBoard callback for logging training metrics
log_dir = "/allah/data/logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

# Train the model with the scaled input
history = model.fit(X_scaled, Y, epochs=200, batch_size=32, validation_split=0.2, callbacks=[tensorboard_callback])

# Plot training and validation loss over epochs
plt.figure(figsize=(10, 5))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.title('Training and Validation Loss Over Epochs')
plt.legend()
plt.grid(True)
plt.show()

# Plot training and validation accuracy over epochs
plt.figure(figsize=(10, 5))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.title('Training and Validation Accuracy Over Epochs')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
X_scaled[-1]

In [ ]:
# Check for NaN and Inf values in X_scaled
print("NaN values in X_scaled:", np.isnan(X_scaled).sum())  # Should be 0
print("Inf values in X_scaled:", np.isinf(X_scaled).sum())  # Should be 0

# Check for NaN and Inf values in Y
print("NaN values in Y:", np.isnan(Y).sum())  # Should be 0
print("Inf values in Y:", np.isinf(Y).sum())  # Should be 0

# Ensure Y is strictly binary (0 or 1)
print("Unique values in Y:", np.unique(Y))  # Should only be [0, 1]
